# AI512: Artificial Intelligence
# Lab 3 — STUDENT QUESTION NOTEBOOK
### Informed Search, Adversarial Search, and Optimization
### Heuristics, A*, IDA*, Beam Search, Minimax, Alpha–Beta, Simulated Annealing, and Genetic Algorithms

**Student Version**

Implement the complete functions specified in each section.

### Instructions
- Do **not** change the given graph, heuristic values, game tree, or experiment settings unless a question explicitly asks you to.
- Implement each required function from its signature, input/output contract, and stated algorithmic requirements.
- Do not change the required function names, parameters, return values, or output format.
- Run the notebook from top to bottom after completing each part.
- Answer written/conceptual questions in Markdown using your own reasoning.
- An unimplemented starter function deliberately raises `NotImplementedError`.


## Setup

Run the cell below first. The lab uses only standard Python libraries plus `numpy` and `matplotlib`.


In [ ]:
from __future__ import annotations
import math
import random
import heapq
from collections import deque
from typing import Dict, List, Tuple, Optional, Any
import numpy as np
import matplotlib.pyplot as plt

random.seed(2026)
np.random.seed(2026)


USE H1 as heuristic function for all the questions whereever necessary

## Graph for Questions 1–5

Use the following weighted directed graph for **Questions 1 to 5**.

- **S** is the **source/start state**.
- **G** is the **goal state**.
- Each edge is directed.
- The numerical value associated with an edge represents its **step cost**.
- Unless stated otherwise, when two nodes have equal priority, use **alphabetical order** for tie-breaking.

```python
GRAPH = {
    'S': {'A': 2, 'B': 4, 'C': 3},
    'A': {'D': 4, 'E': 7},
    'B': {'E': 1, 'F': 5},
    'C': {'F': 2},
    'D': {'G': 5},
    'E': {'G': 3},
    'F': {'G': 2},
    'G': {}
}

## Part 1 — Informed Search and Heuristics

An **informed search algorithm** uses problem-specific knowledge, usually through a heuristic function `h(n)`, to decide which state looks most promising.

For a node `n`:
- `g(n)` = cost from the start state to `n`;
- `h(n)` = estimated cost from `n` to a goal;
- `f(n) = g(n) + h(n)` for A*.

Two important heuristic properties are:
- **Admissible:** `h(n)` never overestimates the true cheapest remaining cost `h*(n)`.
- **Consistent:** for every edge `n -> n'` with cost `c`, `h(n) <= c + h(n')`.

A consistent heuristic is also admissible when `h(goal)=0`.


### Question 1.1 — Diagnose the heuristics

Consider the weighted graph below. Edge costs are shown in the code cell. For each heuristic set, determine whether it is **admissible**, **consistent**, both, or neither. Do not answer from intuition—show at least one inequality that supports your conclusion.

The first heuristic set is intentionally well-behaved; the second contains at least one subtle problem.


In [ ]:
GRAPH = {'S': {'A': 2, 'B': 4, 'C': 3}, 'A': {'D': 4, 'E': 7},
         'B': {'E': 1, 'F': 5}, 'C': {'F': 2}, 'D': {'G': 5},
         'E': {'G': 3}, 'F': {'G': 2}, 'G': {}}
H1 = {'S': 7, 'A': 7, 'B': 4, 'C': 4, 'D': 5, 'E': 3, 'F': 2, 'G': 0}
H2 = {'S': 7, 'A': 9, 'B': 4, 'C': 5, 'D': 5, 'E': 3, 'F': 2, 'G': 0}

def compute_true_costs(graph, goal):
    # Reverse edges: shortest distance FROM any node TO goal becomes a
    # standard shortest-distance problem starting AT goal.
    reverse = {node: [] for node in graph}
    for src, neighbors in graph.items():
        for dst, cost in neighbors.items():
            reverse.setdefault(dst, []).append((src, cost))
    distance = {node: math.inf for node in reverse}
    distance[goal] = 0
    queue = [(0, goal)]
    while queue:
        cost, node = heapq.heappop(queue)
        if cost != distance[node]:  # Old queue entry; ignore it.
            continue
        for predecessor, edge_cost in reverse[node]:
            new_cost = cost + edge_cost
            if new_cost < distance[predecessor]:
                distance[predecessor] = new_cost
                heapq.heappush(queue, (new_cost, predecessor))
    return distance

def check_admissibility(graph, heuristic, goal):
    true_costs = compute_true_costs(graph, goal)
    # Admissible means h(n) <= real cheapest cost for EVERY node.
    violations = [(node, heuristic[node], true_costs[node])
                  for node in graph if heuristic[node] > true_costs[node]]
    return len(violations) == 0, violations, true_costs

def check_consistency(graph, heuristic):
    violations = []
    for src, neighbors in graph.items():
        for dst, step_cost in neighbors.items():
            # Triangle inequality for every directed edge.
            if heuristic[src] > step_cost + heuristic[dst]:
                violations.append((src, dst, heuristic[src], step_cost + heuristic[dst]))
    return len(violations) == 0, violations

def classify_heuristic(graph, heuristic, goal):
    admissible = check_admissibility(graph, heuristic, goal)[0]
    consistent = check_consistency(graph, heuristic)[0]
    if admissible and consistent:
        return 'Both admissible and consistent'
    if admissible:
        return 'Admissible but inconsistent'
    if consistent:
        return 'Consistent but not admissible'
    return 'Neither admissible nor consistent'

for name, heuristic in [('H1', H1), ('H2', H2)]:
    admissible, av, actual = check_admissibility(GRAPH, heuristic, 'G')
    consistent, cv = check_consistency(GRAPH, heuristic)
    print(name, classify_heuristic(GRAPH, heuristic, 'G'))
    print('true costs:', actual, '| admissibility violations:', av, '| consistency violations:', cv)


## Part 2 — Greedy Best-First Search

**Greedy Best-First Search (GBFS)** expands the node with the smallest `h(n)`. It ignores the path cost already paid.

### Your task
Implement `greedy_best_first_search()` so that it returns:
1. the path;
2. total path cost;
3. the expansion order.

Use **alphabetical node order to break ties** in heuristic value.


In [ ]:
def path_cost(graph, path):
    return -1 if not path else sum(graph[u][v] for u, v in zip(path, path[1:]))

def greedy_best_first_search(graph, h, start, goal):
    # Heap priority: (estimated remaining cost, alphabetical name).
    frontier = [(h[start], start)]
    parent = {start: None}  # Remember who first discovered each node.
    visited = set()
    expanded = []
    while frontier:
        _, node = heapq.heappop(frontier)
        if node in visited:
            continue
        visited.add(node)
        expanded.append(node)
        if node == goal:
            path = []
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            return path, path_cost(graph, path), expanded
        for nxt in sorted(graph[node]):
            if nxt not in visited and nxt not in parent:
                parent[nxt] = node
                heapq.heappush(frontier, (h[nxt], nxt))
    return None, -1, expanded

print('Greedy:', greedy_best_first_search(GRAPH, H1, 'S', 'G'))


## Part 3 — A* Search

A* combines cost-so-far and estimated remaining cost:

`f(n) = g(n) + h(n)`

### Your task
Implement graph-search A*. Your implementation must:
- use a priority queue;
- store the best `g` value found so far for each node;
- update a node when a cheaper route is discovered;
- test for the goal when it is **popped for expansion**, not merely when first generated;
- use alphabetical node order as a deterministic tie-break.


In [ ]:
def a_star_search(graph, h, start, goal):
    # Each heap entry is (f, node_name, g). node_name breaks ties.
    frontier = [(h[start], start, 0)]
    best_g = {start: 0}       # Lowest start -> node cost discovered yet.
    parent = {start: None}   # Used to reconstruct the final route.
    expanded = []
    while frontier:
        f, node, popped_g = heapq.heappop(frontier)
        if popped_g != best_g[node]:
            continue         # Skip outdated entries after a cheaper route.
        expanded.append(node)
        if node == goal:     # Goal test happens on POP, never on insertion.
            path = []
            while node is not None:
                path.append(node)
                node = parent[node]
            return path[::-1], popped_g, expanded
        for nxt, step_cost in graph[node].items():
            trial_g = popped_g + step_cost
            if trial_g < best_g.get(nxt, math.inf):
                best_g[nxt] = trial_g
                parent[nxt] = node
                heapq.heappush(frontier, (trial_g + h[nxt], nxt, trial_g))
    return None, -1, expanded

print('A*:', a_star_search(GRAPH, H1, 'S', 'G'))


## Part 4 — IDA* (Iterative Deepening A*)

IDA* combines the low memory use of depth-first search with the heuristic guidance of A*. Instead of limiting **depth**, it repeatedly limits `f(n)=g(n)+h(n)`.

Start with `threshold = h(start)`. During a depth-first iteration:
- prune a node if `f(n) > threshold`;
- remember the smallest exceeded `f` value;
- if the goal is not found, use that minimum exceeded value as the next threshold.

For this task, use `H_IDA`, which is deliberately less informed than `H1` so that multiple threshold iterations are visible.


In [ ]:
H_IDA = {'S': 4, 'A': 5, 'B': 3, 'C': 2, 'D': 4, 'E': 2, 'F': 1, 'G': 0}

def ida_star(graph, h, start, goal):
    threshold = h[start]           # Initial permissible f score.
    thresholds = []
    expansions = 0
    while True:
        thresholds.append(threshold)
        def dfs(node, g, path):
            nonlocal expansions
            f = g + h[node]
            if f > threshold:
                return None, f   # Report smallest f which exceeded bound.
            expansions += 1
            if node == goal:
                return (path[:], g), f
            minimum_excess = math.inf
            for nxt in sorted(graph[node]):
                if nxt in path:   # Avoid cycles within the current DFS path.
                    continue
                result, excess = dfs(nxt, g + graph[node][nxt], path + [nxt])
                if result is not None:
                    return result, excess
                minimum_excess = min(minimum_excess, excess)
            return None, minimum_excess
        result, next_bound = dfs(start, 0, [start])
        if result is not None:
            path, cost = result
            return path, cost, thresholds, expansions
        if next_bound == math.inf:
            return None, -1, thresholds, expansions
        threshold = next_bound

print('IDA*:', ida_star(GRAPH, H_IDA, 'S', 'G'))


## Part 5 — Beam Search

Beam Search is a memory-bounded heuristic search. At each depth, it keeps only the best `k` candidates according to the heuristic and discards the rest.

This makes Beam Search attractive when memory is tight, but it is generally **not complete** and **not optimal**.

### Your task
Implement a level-by-level beam search. At each level:
1. generate all successors of the current beam;
2. sort candidates by `(h(node), node)`;
3. retain only the best `beam_width` candidates.

Return `(path, cost, levels)` where `levels` records the nodes kept after every beam update.


In [ ]:
def beam_search(graph, h, start, goal, beam_width=2):
    if beam_width <= 0:
        raise ValueError('beam_width must be positive')
    # Each state records (path to reach it, total travel cost).
    beam = {start: ([start], 0)}
    kept_levels = []
    while beam:
        if goal in beam:
            path, cost = beam[goal]
            return path, cost, kept_levels
        candidates = {}
        for node, (path, cost) in beam.items():
            for nxt, step in graph[node].items():
                if nxt in path:
                    continue  # Do not loop back along the current path.
                candidate_cost = cost + step
                # If multiple parents generate nxt, keep cheapest path.
                if nxt not in candidates or candidate_cost < candidates[nxt][1]:
                    candidates[nxt] = (path + [nxt], candidate_cost)
        # Local beam keeps only the k best new states AT EACH LEVEL.
        selected = sorted(candidates, key=lambda node: (h[node], node))[:beam_width]
        if not selected:
            break
        kept_levels.append(selected)
        beam = {node: candidates[node] for node in selected}
    return None, -1, kept_levels

for width in [1, 2, 3]:
    print('Beam width', width, ':', beam_search(GRAPH, H1, 'S', 'G', width))


## Part 6 — Search Algorithm Comparison

After implementing Parts 2–5, fill the table programmatically. Use the same `GRAPH` and heuristic where applicable.

For `memory_comment`, use a short phrase such as `stores large frontier`, `linear in depth`, or `bounded by k per level`.


In [ ]:
def build_search_comparison(graph, h_gbfs_astar, h_ida, start='S', goal='G', beam_width=2):
    gp, gc, ge = greedy_best_first_search(graph, h_gbfs_astar, start, goal)
    ap, ac, ae = a_star_search(graph, h_gbfs_astar, start, goal)
    ip, ic, thresholds, ie = ida_star(graph, h_ida, start, goal)
    bp, bc, levels = beam_search(graph, h_gbfs_astar, start, goal, beam_width)
    # Count start + all retained NON-goal nodes (not candidates discarded by cut).
    be = 1 + sum(sum(node != goal for node in level) for level in levels)
    best = compute_true_costs(graph, goal)[start]
    return {
        'Greedy Best-First': {'cost': gc, 'expansions': len(ge),
                              'optimal?': 'yes here' if gc == best else 'no here',
                              'memory_comment': 'stores frontier'},
        'A*': {'cost': ac, 'expansions': len(ae),
               'optimal?': 'yes (admissible h)', 'memory_comment': 'stores large frontier'},
        'IDA*': {'cost': ic, 'expansions': ie,
                 'optimal?': 'yes (admissible h)', 'memory_comment': 'linear in depth'},
        f'Beam Search k={beam_width}': {'cost': bc, 'expansions': be,
                 'optimal?': 'not guaranteed', 'memory_comment': 'bounded by k per level'}
    }

for name, entry in build_search_comparison(GRAPH, H1, H_IDA).items():
    print(name, entry)


## Part 7 — Minimax Search

Minimax is used in deterministic, two-player, zero-sum, perfect-information games.

- **MAX** tries to maximize utility.
- **MIN** tries to minimize utility.
- Terminal nodes contain utility values.

The tree below has three plies: `MAX -> MIN -> MAX -> terminal utility`.


In [ ]:
GAME_TREE = {'R': ['A', 'B', 'C'], 'A': ['A1', 'A2'],
             'B': ['B1', 'B2'], 'C': ['C1', 'C2'],
             'A1': ['A1a', 'A1b'], 'A2': ['A2a', 'A2b'],
             'B1': ['B1a', 'B1b'], 'B2': ['B2a', 'B2b'],
             'C1': ['C1a', 'C1b'], 'C2': ['C2a', 'C2b']}
UTILITY = {'A1a': 3, 'A1b': 5, 'A2a': 6, 'A2b': 9,
           'B1a': 1, 'B1b': 2, 'B2a': 0, 'B2b': -1,
           'C1a': 7, 'C1b': 4, 'C2a': 5, 'C2b': 6}

def minimax(node, maximizing_player=True):
    if node in UTILITY:
        return UTILITY[node]  # A terminal leaf has its own utility.
    values = [minimax(child, not maximizing_player) for child in GAME_TREE[node]]
    return max(values) if maximizing_player else min(values)

def minimax_decision(root='R'):
    best_move, best_value = None, -math.inf
    for child in GAME_TREE[root]:
        value = minimax(child, False)  # After MAX acts, MIN chooses.
        if value > best_value:         # Strict > preserves first action on ties.
            best_move, best_value = child, value
    return best_move, best_value

print('Minimax:', minimax_decision())


## Part 8 — Alpha–Beta Pruning

Alpha–Beta produces the same decision as Minimax but avoids evaluating branches that cannot influence the final choice.

- `alpha` = best value MAX can guarantee so far;
- `beta` = best value MIN can guarantee so far;
- prune when `alpha >= beta`.

### Your task
Implement Alpha–Beta and count how many terminal leaves are evaluated.


In [ ]:
def alpha_beta(node, alpha=-math.inf, beta=math.inf, maximizing_player=True, stats=None):
    if stats is None:
        stats = {'leaves': 0, 'prunes': 0}
    if node in UTILITY:
        stats['leaves'] += 1
        return UTILITY[node]
    if maximizing_player:
        best = -math.inf
        for child in GAME_TREE[node]:
            best = max(best, alpha_beta(child, alpha, beta, False, stats))
            alpha = max(alpha, best)
            if alpha >= beta:
                stats['prunes'] += 1
                break
    else:
        best = math.inf
        for child in GAME_TREE[node]:
            best = min(best, alpha_beta(child, alpha, beta, True, stats))
            beta = min(beta, best)
            if alpha >= beta:
                stats['prunes'] += 1
                break
    return best

def alpha_beta_decision(root='R'):
    stats = {'leaves': 0, 'prunes': 0}
    alpha, best_value, best_child = -math.inf, -math.inf, None
    for child in GAME_TREE[root]:
        value = alpha_beta(child, alpha, math.inf, False, stats)
        if value > best_value:
            best_value, best_child = value, child
        alpha = max(alpha, best_value)
    return best_child, best_value, stats

print('Alpha-beta:', alpha_beta_decision())


## Part 9 — Simulated Annealing on the 8-Queens Problem

We now switch from path search to **optimization**.

Represent an 8-Queens state as a list of length 8. Index = column; value = row. For example:

`[0, 4, 7, 5, 2, 6, 1, 3]`

A solution has **0 attacking queen pairs**.

Simulated Annealing may sometimes accept a worse move. If `delta = new_cost - current_cost > 0`, a common acceptance probability is:

`exp(-delta / T)`

As temperature `T` decreases, accepting worse moves becomes less likely.


In [ ]:
N = 8

def queen_conflicts(state):
    conflicts = 0
    # Every distinct pair (i, j) with i < j gets checked ONCE.
    for i in range(len(state)):
        for j in range(i + 1, len(state)):
            same_row = state[i] == state[j]
            same_diagonal = abs(state[i] - state[j]) == abs(i - j)
            if same_row or same_diagonal:
                conflicts += 1
    return conflicts

def random_neighbor(state):
    candidate = state.copy()    # Do NOT mutate caller's state.
    column = random.randrange(len(candidate))
    rows = [r for r in range(N) if r != candidate[column]]
    candidate[column] = random.choice(rows)  # Exactly one queen changes row.
    return candidate

def temperature_schedule(t, T0=10.0, cooling=0.995):
    return T0 * (cooling ** t)

def accept_worse(delta, T):
    if delta <= 0:
        return 1.0            # Improving/equal move: always accept.
    if T <= 0:
        return 0.0            # No exploration after cooling to zero.
    return math.exp(-delta / T)

def simulated_annealing(max_steps=5000, seed=0, T0=10.0, cooling=0.995):
    random.seed(seed)          # Repeatable results for same parameters.
    current = [random.randrange(N) for _ in range(N)]
    current_cost = queen_conflicts(current)
    best_state, best_cost = current.copy(), current_cost
    history = [best_cost]      # Best-so-far curve NEVER increases.
    for step in range(max_steps):
        if best_cost == 0:
            break
        T = temperature_schedule(step, T0, cooling)
        if T <= 1e-12:        # Effectively zero temperature.
            break
        candidate = random_neighbor(current)
        candidate_cost = queen_conflicts(candidate)
        delta = candidate_cost - current_cost
        if random.random() < accept_worse(delta, T):
            current, current_cost = candidate, candidate_cost
        if current_cost < best_cost:
            best_state, best_cost = current.copy(), current_cost
        history.append(best_cost)
    return best_state, best_cost, history

print('Simulated Annealing:', simulated_annealing(seed=7)[:2])


## Part 10 — Genetic Algorithm on the 8-Queens Problem

A Genetic Algorithm maintains a **population** rather than a single current state.

For this problem:
- chromosome = list of 8 row positions;
- maximum number of non-attacking queen pairs = `C(8,2) = 28`;
- fitness can therefore be `28 - conflicts`;
- selection favors fitter chromosomes;
- crossover combines parents;
- mutation introduces variation.

### Your task
Complete tournament selection, one-point crossover, mutation, and the generation loop.


In [ ]:
MAX_PAIRS = N * (N - 1) // 2

def fitness(chromosome):
    return MAX_PAIRS - queen_conflicts(chromosome)  # Higher is better.

def random_chromosome():
    return [random.randrange(N) for _ in range(N)]

def tournament_select(population, tournament_size=3):
    # Random competitors; strongest survives. Copy protects the population.
    competitors = random.sample(population, min(tournament_size, len(population)))
    return max(competitors, key=fitness).copy()

def one_point_crossover(p1, p2):
    point = random.randrange(1, len(p1))  # Cut is strictly inside the genes.
    return p1[:point] + p2[point:], p2[:point] + p1[point:]

def mutate(chromosome, mutation_rate=0.10):
    child = chromosome.copy()
    for index, old_row in enumerate(child):
        if random.random() < mutation_rate:
            child[index] = random.choice([r for r in range(N) if r != old_row])
    return child

def genetic_algorithm(pop_size=100, generations=500, mutation_rate=0.10, seed=0):
    if pop_size < 2:
        raise ValueError('pop_size must be at least 2')
    random.seed(seed)
    population = [random_chromosome() for _ in range(pop_size)]
    best = max(population, key=fitness).copy()
    best_score = fitness(best)
    history = [best_score]
    if best_score == MAX_PAIRS:
        return best, best_score, history, 0
    for generation in range(1, generations + 1):
        elite = max(population, key=fitness).copy()
        next_population = [elite]  # Elitism: never lose this generation's best.
        while len(next_population) < pop_size:
            parent1 = tournament_select(population)
            parent2 = tournament_select(population)
            child1, child2 = one_point_crossover(parent1, parent2)
            for child in (child1, child2):
                if len(next_population) < pop_size:
                    next_population.append(mutate(child, mutation_rate))
        population = next_population
        generation_best = max(population, key=fitness)
        if fitness(generation_best) > best_score:
            best = generation_best.copy()
            best_score = fitness(best)
        history.append(best_score)
        if best_score == MAX_PAIRS:
            return best, best_score, history, generation
    return best, best_score, history, generations

print('Genetic Algorithm:', genetic_algorithm(seed=7)[:2])


## Graded-syllabus supplement: **Local Beam Search** (8-Queens)

**Important difference:** Part 5 in the supplied notebook is *level-by-level beam search on a graph*. **Local Beam Search** typically starts with `k` current candidate states, generates all their local neighbours, and keeps the `k` best successors across the entire pool at each iteration. For 8-Queens we minimize conflicts. The implementation below provides the distinct Local Beam Search algorithm for your graded syllabus.

In [ ]:

def local_beam_search(k=4, max_steps=500, seed=7):
    """Return (best_8_queen_state, conflicts, best_history)."""
    if k <= 0:
        raise ValueError('k must be positive')
    random.seed(seed)
    states = [random_chromosome() for _ in range(k)]  # k random starting states.
    best = min(states, key=queen_conflicts).copy()
    history = [queen_conflicts(best)]
    for _ in range(max_steps):
        if queen_conflicts(best) == 0:
            break
        all_neighbors = []
        for state in states:
            # All N * (N-1) one-queen moves from each current state.
            for column in range(N):
                for row in range(N):
                    if row == state[column]:
                        continue
                    neighbor = state.copy()
                    neighbor[column] = row
                    all_neighbors.append(neighbor)
        # A plain deterministic local beam can stagnate; deduplicate states.
        unique = list({tuple(s): s for s in all_neighbors}.values())
        unique.sort(key=lambda s: (queen_conflicts(s), tuple(s)))
        states = unique[:k]  # Keep only the k best successors globally.
        if queen_conflicts(states[0]) < queen_conflicts(best):
            best = states[0].copy()
        history.append(queen_conflicts(best))
    return best, queen_conflicts(best), history

print('Local Beam Search:', local_beam_search()[:2])


## Part 11 — Simulated Annealing vs Genetic Algorithm

Use repeated trials so that your conclusion does not depend on one lucky seed.

### Experimental task
Run **50 seeds** for both algorithms and collect:
- success rate;
- average final conflicts;
- median final conflicts;
- average runtime or iteration/generation count;
- best and worst final conflicts.

Then plot the distribution of final conflict counts.


In [ ]:
def run_sa_ga_experiment(num_seeds=50):
    sa_conflicts, ga_conflicts, sa_iterations, ga_generations = [], [], [], []
    for seed in range(num_seeds):
        sa_state, sa_cost, sa_hist = simulated_annealing(seed=seed)
        ga_state, ga_score, ga_hist, generations_used = genetic_algorithm(seed=seed)
        sa_conflicts.append(sa_cost)
        ga_conflicts.append(MAX_PAIRS - ga_score)
        sa_iterations.append(len(sa_hist) - 1)  # Excludes the initial state.
        ga_generations.append(generations_used)
    return sa_conflicts, ga_conflicts, sa_iterations, ga_generations

def summarize_trials(name, conflicts, effort, effort_label):
    print(f'\n{name}:')
    print(f'  Success rate: {sum(c == 0 for c in conflicts) / len(conflicts):.1%}')
    print(f'  Mean conflicts: {np.mean(conflicts):.3f}; median: {np.median(conflicts):.3f}')
    print(f'  Average {effort_label}: {np.mean(effort):.2f}')
    print(f'  Best conflicts: {min(conflicts)}; worst: {max(conflicts)}')

def plot_conflict_distribution(sa_final_conflicts, ga_final_conflicts):
    all_values = sa_final_conflicts + ga_final_conflicts
    bins = np.arange(min(all_values) - 0.5, max(all_values) + 1.5, 1)
    plt.figure(figsize=(8, 4))
    plt.hist([sa_final_conflicts, ga_final_conflicts], bins=bins,
             label=['Simulated Annealing', 'Genetic Algorithm'], alpha=.75)
    plt.xlabel('Final number of queen conflicts (lower is better)')
    plt.ylabel('Frequency across seeds')
    plt.legend()
    plt.tight_layout()
    plt.show()

sa_cs, ga_cs, sa_it, ga_gens = run_sa_ga_experiment(50)
summarize_trials('Simulated Annealing', sa_cs, sa_it, 'SA iterations')
summarize_trials('Genetic Algorithm', ga_cs, ga_gens, 'GA generations')
plot_conflict_distribution(sa_cs, ga_cs)


## Part 12 — Integrated Challenge Questions

These questions are intended to test understanding rather than memorized definitions.

**Q1.** Two admissible heuristics satisfy `h2(n) >= h1(n)` for every node. Under standard A* graph-search assumptions, which heuristic is expected to expand no more nodes, and why?

**Q2.** Construct a small graph in which Greedy Best-First Search reaches a very expensive goal while A* finds a much cheaper path.

**Q3.** A Beam Search with width `k=2` fails, but width `k=3` succeeds. Does that prove width `3` is complete? Explain.

**Q4.** Why is IDA* often attractive for puzzles such as the 15-puzzle even though it re-expands states?

**Q5.** In Minimax, why can MAX not simply choose the child with the largest immediate evaluation if there are MIN moves below it?

**Q6.** Alpha–Beta prunes a branch. Does that mean the pruned branch has no legal moves, or something else? State exactly what the prune means.

**Q7.** Give an example of poor move ordering that causes Alpha–Beta to perform almost like ordinary Minimax.

**Q8.** Simulated Annealing accepts a worse state at high temperature. Explain why this is not a “bug.”

**Q9.** A Genetic Algorithm reaches identical chromosomes across the entire population but has not found a solution. Name the phenomenon and give two remedies.

**Q10.** Compare A* and Genetic Algorithms with respect to what a “state,” “successor,” and “solution” mean. Why is GA usually not described as a shortest-path algorithm?

**Q11.** For each of the following, choose the most appropriate algorithm from this lab and justify your choice: (a) shortest route with a trustworthy heuristic, (b) chess-like two-player game, (c) huge-memory-limited puzzle, (d) continuous/noisy optimization landscape, (e) population-based combinatorial optimization.

**Q12.** Suppose an A* heuristic is admissible but inconsistent. What implementation detail becomes especially important in graph search?


Write your answers to **Q1–Q12** from the previous section below. Add extra markdown cells if you need more space.


## Part 12 — Suggested reasoning answers (study reference)

1. Under usual admissible/consistent and tie-handling conditions, a more informative dominating heuristic generally expands fewer states; on equal `f` contours, expansion counts can depend on tie-breaking. Dominance alone does not guarantee a literal subset of expansions under arbitrary tie breaks.
2. Example: S→A (1), A→G (100), S→B (2), B→G (2); choose `h(A)=0, h(B)=2, h(G)=0`. Greedy chooses A first then G, costing 101; A* finds S→B→G costing 4 (with suitable tie-breaking). The heuristic is not required to be accurate for illustrating Greedy's failure; here it is admissible.
3. No: finite k may discard the only viable path in another graph or at another depth.
4. IDA* uses memory proportional to current search depth rather than a large A* frontier, although it can re-expand states.
5. MIN is adversarial: MAX must evaluate the worst response after each candidate move.
6. It means that branch cannot improve the ancestor's decision given the existing alpha/beta bounds, not that the moves are illegal.
7. At MAX nodes, exploring the weakest moves first and at MIN nodes the highest-valued replies first can yield few cutoffs.
8. Worse moves allow escape from local minima; cooling gradually reduces this exploration.
9. Premature convergence/loss of diversity. Remedies: increase mutation and inject random immigrants/new chromosomes.
10. A* states are graph locations, successors are graph edges, and solution is a goal-reaching route (often cheapest). GA individuals encode candidate solutions, successors are crossover/mutation offspring, and solution is a high-fitness chromosome; it does not inherently minimize path cost.
11. (a) A*, (b) Minimax / Alpha–Beta, (c) IDA*, (d) Simulated Annealing, (e) Genetic Algorithm.
12. Reopening a previously expanded node if a cheaper `g` path is found is crucial; an irrevocable closed set may break optimality with inconsistent heuristics.
